# So sánh bộ lọc biên và khử nhiễu
Cùng ảnh sạch, cùng seed và cùng ảnh nhiễu cho mọi bộ lọc. Có Gaussian σ=15, muối tiêu 5%, JPEG quality=25. Biên có GT 4 góc và cùng bóng. Đây là cấu hình tham khảo cố định, cần tuning trên validation trước khi so test.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/pipeline.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Đặt working directory trong project đã giải nén/clone.")
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import cv2
import matplotlib.pyplot as plt


In [ ]:
import subprocess
import pandas as pd
CLEAN = Path("data/raw_clean")
from src.utils import image_files
if not image_files(CLEAN):
    CLEAN = Path("experiments/smoke/clean")
    if not image_files(CLEAN):
        subprocess.run([sys.executable,"-m","scripts.make_demo_data","--output",str(CLEAN)],check=True)
    print("Đang dùng ảnh demo, không phải benchmark.")
subprocess.run([sys.executable,"-m","scripts.compare_filters","--clean",str(CLEAN),"--limit","5"],check=True)
noise = pd.read_csv("report/tables/denoise_comparison.csv")
edges = pd.read_csv("report/tables/edge_comparison.csv")
display(noise.groupby(["noise","filter"])[["psnr","ssim","elapsed_seconds"]].agg(["mean","std"]))


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
for ax,metric in zip(axes,["psnr","ssim"]):
    noise.groupby(["noise","filter"])[metric].mean().unstack(0).plot.bar(ax=ax,title=metric,rot=35)
plt.tight_layout()
fig.savefig("report/figures/denoise_comparison.png",dpi=160)


## Độ ổn định viền trang
Báo detection rate cùng corner error: sai số nhỏ trên vài ảnh tìm được không đồng nghĩa bộ lọc tốt nhất. Cần kiểm tra failure cases. Trường hợp đạo hàm có biên kép hoặc bóng gần viền có thể làm contour sai.

In [ ]:
summary = edges.groupby(["condition","filter"]).agg(
    detection_rate=("detected","mean"), mean_corner_error_px=("corner_error_px","mean"),
    std_corner_error_px=("corner_error_px","std"), elapsed_seconds=("elapsed_seconds","mean"))
display(summary)
summary.to_csv("report/tables/edge_summary.csv")
fig,axes=plt.subplots(1,2,figsize=(14,5))
summary["detection_rate"].unstack(0).plot.bar(ax=axes[0],title="Detection rate",rot=35)
summary["mean_corner_error_px"].unstack(0).plot.bar(ax=axes[1],title="Corner error (detected only)",rot=35)
plt.tight_layout()
fig.savefig("report/figures/edge_comparison.png",dpi=160)


## Hình minh họa cùng một ảnh
Quan sát cả độ sạch nền và bảo toàn nét/dấu chữ. PSNR cao chưa chắc OCR tốt hơn.

In [ ]:
from src.utils import read_rgb
folder=Path("report/figures/filters")
fig,axes=plt.subplots(2,4,figsize=(16,10))
for ax,name in zip(axes.flat,["input","mean","gaussian","median","bilateral","nlm","wiener"]):
    ax.imshow(read_rgb(folder/f"salt_pepper_{name}.png")); ax.set_title(name); ax.axis("off")
axes.flat[-1].axis("off")
plt.tight_layout()
